In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.contingency_tables import mcnemar

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             roc_curve, auc, brier_score_loss, confusion_matrix)
from sklearn.calibration import calibration_curve

print("Starting Master Script for Domain Shift Paper (Final Restored Version)...")

df_ger = pd.read_csv('diabetes_GERMANY.csv')
df_us = pd.read_csv('diabetes_prediction_us.csv')

df_us['Outcome'] = df_us['Diabetes_012'].apply(lambda x: 1 if x > 0 else 0)

df_ger['BloodPressure'] = df_ger['BloodPressure'].replace(0, df_ger['BloodPressure'].median())
df_ger['BMI'] = df_ger['BMI'].replace(0, df_ger['BMI'].median())

df_ger['HighBP'] = df_ger['BloodPressure'].apply(lambda x: 1 if x >= 80 else 0)

bins = [0, 24, 29, 34, 39, 44, 49, 54, 59, 64, 69, 74, 79, 150]
labels = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0]
df_ger['Age_Cat'] = pd.cut(df_ger['Age'], bins=bins, labels=labels).astype(float)
df_ger['Age'] = df_ger['Age_Cat']

features = ['BMI', 'Age', 'HighBP']

df_us_sampled = df_us.sample(n=6000, random_state=42)

X_us = df_us_sampled[features]
y_us = df_us_sampled['Outcome'].values
X_ger = df_ger[features]
y_ger = df_ger['Outcome'].values

X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(
    X_ger, y_ger, test_size=0.2, random_state=42, stratify=y_ger
)

scaler_cross = StandardScaler()
X_us_scaled = scaler_cross.fit_transform(X_us)
X_test_g_scaled_cross = scaler_cross.transform(X_test_g)

scaler_local = StandardScaler()
X_train_g_scaled = scaler_local.fit_transform(X_train_g)
X_test_g_scaled = scaler_local.transform(X_test_g)

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train_g_scaled, y_train_g)
y_pred_dummy = dummy.predict(X_test_g_scaled)
base_acc = accuracy_score(y_test_g, y_pred_dummy)
print(f"Majority Class Baseline Accuracy: {base_acc:.4f}\n")

def compute_bootstrap_ci(y_true, y_pred, metric_func, n_bootstraps=1000):
    rng = np.random.RandomState(42)
    scores = []
    for _ in range(n_bootstraps):
        indices = rng.randint(0, len(y_pred), len(y_pred))
        if len(np.unique(y_true[indices])) < 2:
            continue
        scores.append(metric_func(y_true[indices], y_pred[indices]))
    return np.percentile(scores, 2.5), np.percentile(scores, 97.5)

def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else 0

models = {
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, max_depth=5),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=5),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results = []
roc_data_cross, roc_data_local = {}, {}
cms_cross, cms_local = {}, {}
predictions_cross, predictions_local = {}, {}
prob_cross_dict = {}

for name, model in models.items():
    model.fit(X_us_scaled, y_us)
    y_pred_cross = model.predict(X_test_g_scaled_cross)
    predictions_cross[name] = y_pred_cross
    cms_cross[name] = confusion_matrix(y_test_g, y_pred_cross)
    
    acc_c = accuracy_score(y_test_g, y_pred_cross)
    acc_c_lower, acc_c_upper = compute_bootstrap_ci(y_test_g, y_pred_cross, accuracy_score)
    spec_c = specificity_score(y_test_g, y_pred_cross)
    
    auc_cross, brier_cross = np.nan, np.nan
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X_test_g_scaled_cross)[:, 1]
        prob_cross_dict[name] = probs
        fpr, tpr, _ = roc_curve(y_test_g, probs)
        auc_cross = auc(fpr, tpr)
        roc_data_cross[name] = (fpr, tpr, auc_cross)
        brier_cross = brier_score_loss(y_test_g, probs)
        
    results.append({
        'Model': name, 'Setup': 'Cross-Continent',
        'Accuracy': acc_c,
        'Accuracy_CI': f"{acc_c:.3f} ({acc_c_lower:.2f}-{acc_c_upper:.2f})",
        'Precision': f"{precision_score(y_test_g, y_pred_cross, zero_division=0):.3f}",
        'Recall': f"{recall_score(y_test_g, y_pred_cross, zero_division=0):.3f}",
        'Specificity': f"{spec_c:.3f}",
        'F1-Score': f"{f1_score(y_test_g, y_pred_cross, zero_division=0):.3f}",
        'AUC': f"{auc_cross:.3f}" if not np.isnan(auc_cross) else "-",
        'Brier Score': f"{brier_cross:.3f}" if not np.isnan(brier_cross) else "-"
    })
    
    model.fit(X_train_g_scaled, y_train_g)
    y_pred_local = model.predict(X_test_g_scaled)
    predictions_local[name] = y_pred_local
    cms_local[name] = confusion_matrix(y_test_g, y_pred_local)
    
    acc_l = accuracy_score(y_test_g, y_pred_local)
    acc_l_lower, acc_l_upper = compute_bootstrap_ci(y_test_g, y_pred_local, accuracy_score)
    spec_l = specificity_score(y_test_g, y_pred_local)
    
    auc_local, brier_local = np.nan, np.nan
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X_test_g_scaled)[:, 1]
        fpr, tpr, _ = roc_curve(y_test_g, probs)
        auc_local = auc(fpr, tpr)
        roc_data_local[name] = (fpr, tpr, auc_local)
        brier_local = brier_score_loss(y_test_g, probs)
        
    results.append({
        'Model': name, 'Setup': 'Local',
        'Accuracy': acc_l,
        'Accuracy_CI': f"{acc_l:.3f} ({acc_l_lower:.2f}-{acc_l_upper:.2f})",
        'Precision': f"{precision_score(y_test_g, y_pred_local, zero_division=0):.3f}",
        'Recall': f"{recall_score(y_test_g, y_pred_local, zero_division=0):.3f}",
        'Specificity': f"{spec_l:.3f}",
        'F1-Score': f"{f1_score(y_test_g, y_pred_local, zero_division=0):.3f}",
        'AUC': f"{auc_local:.3f}" if not np.isnan(auc_local) else "-",
        'Brier Score': f"{brier_local:.3f}" if not np.isnan(brier_local) else "-"
    })

print("--- McNemar's Exact Test Results (Local vs Cross) ---")
for name in models.keys():
    local_correct = (predictions_local[name] == y_test_g)
    cross_correct = (predictions_cross[name] == y_test_g)
    table = [[np.sum(local_correct & cross_correct), np.sum(local_correct & ~cross_correct)],
             [np.sum(~local_correct & cross_correct), np.sum(~local_correct & ~cross_correct)]]
    result = mcnemar(table, exact=True)
    print(f"{name}: exact p-value = {result.pvalue:.5e}")

res_df = pd.DataFrame(results)
res_df.drop(columns=['Accuracy']).to_csv("final_metrics_complete_with_CIs.csv", index=False)

plt.style.use('seaborn-v0_8-paper')

plt.figure(figsize=(10, 5))
sns.barplot(data=res_df, x='Model', y='Accuracy', hue='Setup', palette='mako')
plt.axhline(y=base_acc, color='r', linestyle='--', label=f'Baseline ({base_acc:.2f})')
plt.title('Figure 1: Accuracy Comparison (Local vs Cross-Continent)', fontweight='bold')
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('figure1_accuracy.png', dpi=300)
plt.close()

plt.figure(figsize=(8, 6))
ax = plt.subplot2grid((1, 1), (0, 0))
ax.plot([0, 1], [0, 1], "k:", label="Perfectly calibrated")
for name in prob_cross_dict.keys():
    prob_pos = prob_cross_dict[name]
    fraction_of_positives, mean_predicted_value = calibration_curve(y_test_g, prob_pos, n_bins=10)
    ax.plot(mean_predicted_value, fraction_of_positives, "s-", label=name)
ax.set_ylabel("Fraction of positives")
ax.set_xlabel("Mean predicted value")
ax.set_title('Figure 2: Calibration Curves (Cross-Continent Models)', fontweight='bold')
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig('figure2_calibration.png', dpi=300)
plt.close()

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
sns.kdeplot(data=X_us, x='BMI', fill=True, label='US', color='blue')
sns.kdeplot(data=X_ger, x='BMI', fill=True, label='GER', color='red')
plt.title('BMI Distribution', fontweight='bold')
plt.legend()
plt.subplot(1, 2, 2)
sns.kdeplot(data=X_us, x='Age', fill=True, label='US', color='blue')
sns.kdeplot(data=X_ger, x='Age', fill=True, label='GER', color='red')
plt.title('Age Distribution Bins', fontweight='bold')
plt.xlabel('CDC Age Categories')
plt.legend()
plt.tight_layout()
plt.savefig('figure3_covariate_shift.png', dpi=300)
plt.close()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
for name in roc_data_cross.keys():
    fpr, tpr, roc_auc = roc_data_cross[name]
    ax1.plot(fpr, tpr, lw=2, label=f'{name} (AUC={roc_auc:.2f})')
ax1.plot([0, 1], [0, 1], color='black', linestyle='--')
ax1.set_title('Cross-Continent (US -> GER)', fontweight='bold')
ax1.legend(loc="lower right")

for name in roc_data_local.keys():
    fpr, tpr, roc_auc = roc_data_local[name]
    ax2.plot(fpr, tpr, lw=2, label=f'{name} (AUC={roc_auc:.2f})')
ax2.plot([0, 1], [0, 1], color='black', linestyle='--')
ax2.set_title('Local (GER -> GER)', fontweight='bold')
ax2.legend(loc="lower right")
plt.tight_layout()
plt.savefig('figure4_roc.png', dpi=300)
plt.close()

print("\nComplete! Accuracy Chart restored. Calibration added. Figures generated 1 through 4.")

Starting Master Script for Domain Shift Paper (Final Restored Version)...
Majority Class Baseline Accuracy: 0.6575

--- McNemar's Exact Test Results (Local vs Cross) ---
Naive Bayes: exact p-value = 2.54305e-01
Logistic Regression: exact p-value = 3.44159e-02
AdaBoost: exact p-value = 5.04870e-02
Random Forest: exact p-value = 1.06941e-02
Decision Tree: exact p-value = 3.42357e-02
Gradient Boosting: exact p-value = 9.64781e-05

Complete! Accuracy Chart restored. Calibration added. Figures generated 1 through 4.
